# Project 2: PCA for Dimensionality Reduction

## Introduction

Welcome to Project 2! In this notebook, we'll explore **Principal Component Analysis (PCA)**, one of the most fundamental techniques in machine learning and data science.

### What is PCA?

PCA is a dimensionality reduction technique that finds the directions (principal components) along which the data varies the most. It's essentially SVD applied to mean-centered data!

### Why PCA?

- **Dimensionality Reduction**: Reduce 1000 features to 10 while preserving most information
- **Visualization**: Project high-dimensional data to 2D/3D for plotting
- **Noise Reduction**: Filter out noise by keeping only top components
- **Preprocessing**: Speed up ML algorithms by reducing features
- **Feature Engineering**: Create uncorrelated features from correlated ones

### The Math Behind PCA

Given data matrix $X$ (n samples × d features):

1. **Center the data**: $\tilde{X} = X - \text{mean}(X)$
2. **Compute SVD**: $\tilde{X} = U\Sigma V^T$
3. **Principal components** are columns of $V$
4. **Project data**: $Z = \tilde{X}V_k$ (keep first k components)

Alternative formulation:
- Covariance matrix: $C = \frac{1}{n-1}\tilde{X}^T\tilde{X}$
- PCA finds eigenvectors of $C$ (same as $V$ from SVD!)

### Learning Objectives

1. Implement PCA from scratch using SVD
2. Apply PCA to real datasets (MNIST digits)
3. Visualize high-dimensional data in 2D/3D
4. Understand explained variance and choosing k
5. Compare with scikit-learn's PCA

---

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as SklearnPCA
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# Set random seed
np.random.seed(42)

# Matplotlib config
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

## Part 1: Understanding PCA with a Simple 2D Example

In [ ]:
# Generate 2D correlated data
np.random.seed(42)
n_samples = 300

# Create correlated 2D data
mean = [0, 0]
cov = [[3, 2], [2, 2]]  # Covariance matrix
data_2d = np.random.multivariate_normal(mean, cov, n_samples)

# Visualize
plt.figure(figsize=(8, 8))
plt.scatter(data_2d[:, 0], data_2d[:, 1], alpha=0.6, s=30)
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Original 2D Data (Correlated)', fontsize=14, fontweight='bold')
plt.axis('equal')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Data shape: {data_2d.shape}")
print(f"\nData mean: {np.mean(data_2d, axis=0)}")
print(f"\nCovariance matrix:\n{np.cov(data_2d.T)}")

## Part 2: Implement PCA from Scratch

In [ ]:
class PCA:
    """
    Principal Component Analysis implemented from scratch using SVD.
    """
    
    def __init__(self, n_components=None):
        """
        Args:
            n_components: Number of components to keep. If None, keep all.
        """
        self.n_components = n_components
        self.components_ = None
        self.mean_ = None
        self.explained_variance_ = None
        self.explained_variance_ratio_ = None
        
    def fit(self, X):
        """
        Fit PCA on data X.
        
        Args:
            X: Data matrix (n_samples, n_features)
        """
        # Step 1: Center the data
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_
        
        # Step 2: Compute SVD
        U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
        
        # Step 3: Principal components are right singular vectors
        self.components_ = Vt
        
        # Step 4: Compute explained variance
        # Variance explained by each component
        self.explained_variance_ = (S ** 2) / (X.shape[0] - 1)
        
        # Ratio of variance explained
        total_var = np.sum(self.explained_variance_)
        self.explained_variance_ratio_ = self.explained_variance_ / total_var
        
        # Keep only n_components if specified
        if self.n_components is not None:
            self.components_ = self.components_[:self.n_components]
            self.explained_variance_ = self.explained_variance_[:self.n_components]
            self.explained_variance_ratio_ = self.explained_variance_ratio_[:self.n_components]
        
        return self
    
    def transform(self, X):
        """
        Project X onto principal components.
        
        Args:
            X: Data matrix (n_samples, n_features)
            
        Returns:
            X_transformed: Projected data (n_samples, n_components)
        """
        X_centered = X - self.mean_
        return X_centered @ self.components_.T
    
    def fit_transform(self, X):
        """
        Fit and transform in one step.
        """
        self.fit(X)
        return self.transform(X)
    
    def inverse_transform(self, X_transformed):
        """
        Reconstruct data from principal components.
        """
        return X_transformed @ self.components_ + self.mean_

print("✓ PCA class implemented!")

### Apply PCA to 2D Data

In [ ]:
# Fit PCA
pca = PCA(n_components=2)
pca.fit(data_2d)

print("Principal Components (rows are PCs):")
print(pca.components_)
print(f"\nExplained variance: {pca.explained_variance_}")
print(f"\nExplained variance ratio: {pca.explained_variance_ratio_}")
print(f"\nTotal variance explained: {np.sum(pca.explained_variance_ratio_):.4f}")

# Visualize principal components
plt.figure(figsize=(8, 8))
plt.scatter(data_2d[:, 0], data_2d[:, 1], alpha=0.6, s=30, label='Data')

# Draw principal component vectors
mean = pca.mean_
for i, (comp, var) in enumerate(zip(pca.components_, pca.explained_variance_)):
    # Scale by sqrt of eigenvalue for visualization
    comp_scaled = comp * np.sqrt(var) * 3
    plt.arrow(mean[0], mean[1], comp_scaled[0], comp_scaled[1], 
             head_width=0.3, head_length=0.2, fc=f'C{i+1}', ec=f'C{i+1}', 
             linewidth=3, label=f'PC{i+1} ({pca.explained_variance_ratio_[i]*100:.1f}%)')

plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Principal Components Visualization', fontsize=14, fontweight='bold')
plt.axis('equal')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Project Data onto Principal Components

In [ ]:
# Transform data
data_transformed = pca.transform(data_2d)

# Visualize transformed data
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Original space
ax1.scatter(data_2d[:, 0], data_2d[:, 1], alpha=0.6, s=30)
ax1.set_xlabel('Feature 1', fontsize=12)
ax1.set_ylabel('Feature 2', fontsize=12)
ax1.set_title('Original Space', fontsize=14, fontweight='bold')
ax1.axis('equal')
ax1.grid(alpha=0.3)

# Principal component space
ax2.scatter(data_transformed[:, 0], data_transformed[:, 1], alpha=0.6, s=30, color='orange')
ax2.set_xlabel('PC 1', fontsize=12)
ax2.set_ylabel('PC 2', fontsize=12)
ax2.set_title('Principal Component Space (Uncorrelated!)', fontsize=14, fontweight='bold')
ax2.axis('equal')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Covariance in PC space (should be diagonal!):")
print(np.cov(data_transformed.T))

## Part 3: PCA on MNIST Digits

Now let's apply PCA to a real-world dataset: MNIST handwritten digits!

In [ ]:
# Load MNIST digits (subset)
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data  # 8x8 images flattened to 64 features
y = digits.target

print(f"Dataset shape: {X.shape}")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]} (8x8 images flattened)")
print(f"Number of classes: {len(np.unique(y))}")

# Visualize some digits
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(X[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'Label: {y[i]}')
    ax.axis('off')
plt.suptitle('Sample MNIST Digits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Apply PCA for Visualization (64D → 2D)

In [ ]:
# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA to reduce to 2D
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

print(f"Original dimensions: {X.shape[1]}")
print(f"Reduced dimensions: {X_pca_2d.shape[1]}")
print(f"\nVariance explained by 2 components: {np.sum(pca_2d.explained_variance_ratio_)*100:.2f}%")

# Visualize in 2D
plt.figure(figsize=(12, 10))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], 
                     c=y, cmap='tab10', alpha=0.7, s=30, edgecolors='k', linewidth=0.5)
plt.xlabel(f'PC 1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
plt.ylabel(f'PC 2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
plt.title('MNIST Digits in 2D (PCA)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Digit Class')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 3D Visualization

In [ ]:
# Apply PCA to reduce to 3D
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_scaled)

# 3D plot
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(X_pca_3d[:, 0], X_pca_3d[:, 1], X_pca_3d[:, 2],
                    c=y, cmap='tab10', alpha=0.6, s=20, edgecolors='k', linewidth=0.3)

ax.set_xlabel(f'PC 1 ({pca_3d.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11)
ax.set_ylabel(f'PC 2 ({pca_3d.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)
ax.set_zlabel(f'PC 3 ({pca_3d.explained_variance_ratio_[2]*100:.1f}%)', fontsize=11)
ax.set_title('MNIST Digits in 3D (PCA)', fontsize=14, fontweight='bold')

plt.colorbar(scatter, label='Digit Class', pad=0.1)
plt.tight_layout()
plt.show()

print(f"Variance explained by 3 components: {np.sum(pca_3d.explained_variance_ratio_)*100:.2f}%")

## Part 4: Choosing the Number of Components

In [ ]:
# Fit PCA with all components
pca_full = PCA()
pca_full.fit(X_scaled)

# Cumulative explained variance
cumsum = np.cumsum(pca_full.explained_variance_ratio_)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Individual explained variance
ax1.bar(range(1, len(pca_full.explained_variance_ratio_)+1), 
        pca_full.explained_variance_ratio_, alpha=0.7, color='steelblue')
ax1.set_xlabel('Principal Component', fontsize=12)
ax1.set_ylabel('Explained Variance Ratio', fontsize=12)
ax1.set_title('Explained Variance by Component', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Cumulative explained variance
ax2.plot(range(1, len(cumsum)+1), cumsum, 'o-', linewidth=2, markersize=5)
ax2.axhline(y=0.80, color='r', linestyle='--', label='80% variance')
ax2.axhline(y=0.90, color='g', linestyle='--', label='90% variance')
ax2.axhline(y=0.95, color='b', linestyle='--', label='95% variance')
ax2.set_xlabel('Number of Components', fontsize=12)
ax2.set_ylabel('Cumulative Explained Variance', fontsize=12)
ax2.set_title('Cumulative Explained Variance', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Find number of components for different variance thresholds
for threshold in [0.80, 0.90, 0.95]:
    n_comp = np.argmax(cumsum >= threshold) + 1
    print(f"Components needed for {threshold*100:.0f}% variance: {n_comp}")

## Part 5: Visualizing Principal Components

In [ ]:
# Visualize the first 12 principal components
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i < len(pca_full.components_):
        # Reshape component to 8x8 image
        component_img = pca_full.components_[i].reshape(8, 8)
        
        ax.imshow(component_img, cmap='RdBu_r', aspect='auto')
        ax.set_title(f'PC {i+1}\n({pca_full.explained_variance_ratio_[i]*100:.1f}% var)', 
                    fontsize=10)
        ax.axis('off')

plt.suptitle('First 12 Principal Components (Eigendigits)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 6: Image Reconstruction from Components

In [ ]:
# Reconstruct images using different numbers of components
n_components_list = [5, 10, 20, 30, 40, 64]

# Pick a sample digit
sample_idx = 42
original_digit = X_scaled[sample_idx]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# Original
axes[0].imshow(original_digit.reshape(8, 8), cmap='gray')
axes[0].set_title(f'Original (Digit {y[sample_idx]})', fontsize=11, fontweight='bold')
axes[0].axis('off')

# Reconstructions
for idx, n_comp in enumerate(n_components_list, 1):
    # Project and reconstruct
    pca_temp = PCA(n_components=n_comp)
    X_proj = pca_temp.fit_transform(X_scaled)
    X_reconstructed = pca_temp.inverse_transform(X_proj)
    
    # Show reconstruction
    reconstructed_digit = X_reconstructed[sample_idx]
    axes[idx].imshow(reconstructed_digit.reshape(8, 8), cmap='gray')
    
    # Calculate reconstruction error
    mse = np.mean((original_digit - reconstructed_digit)**2)
    var_explained = np.sum(pca_temp.explained_variance_ratio_) * 100
    
    axes[idx].set_title(f'{n_comp} components\n{var_explained:.1f}% var | MSE: {mse:.3f}', 
                       fontsize=10)
    axes[idx].axis('off')

# Hide last subplot if needed
if len(n_components_list) < 7:
    axes[-1].axis('off')

plt.suptitle('Image Reconstruction with Different Numbers of Components', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 7: Compare with Scikit-Learn's PCA

In [ ]:
# Our implementation
our_pca = PCA(n_components=10)
X_our = our_pca.fit_transform(X_scaled)

# Scikit-learn's implementation
sklearn_pca = SklearnPCA(n_components=10)
X_sklearn = sklearn_pca.fit_transform(X_scaled)

# Compare results
print("=== Comparison: Our PCA vs Scikit-Learn PCA ===")
print(f"\nExplained variance ratio (ours):")
print(our_pca.explained_variance_ratio_)
print(f"\nExplained variance ratio (sklearn):")
print(sklearn_pca.explained_variance_ratio_)

print(f"\nDifference in explained variance ratios:")
print(np.abs(our_pca.explained_variance_ratio_ - sklearn_pca.explained_variance_ratio_))

# The transformed data might differ by sign (PCs are defined up to sign)
print(f"\nMax absolute difference in transformed data: {np.max(np.abs(np.abs(X_our) - np.abs(X_sklearn))):.10f}")
print("\n✓ Our implementation matches scikit-learn!")

## Part 8: PCA for Noise Reduction

In [ ]:
# Add noise to a digit
sample_idx = 10
clean_digit = X[sample_idx]

# Add Gaussian noise
noise = np.random.normal(0, 4, clean_digit.shape)
noisy_digit = clean_digit + noise
noisy_digit = np.clip(noisy_digit, 0, 16)  # Clip to valid range

# Standardize
noisy_scaled = scaler.transform([noisy_digit])[0]

# Denoise using PCA (keep only top components)
pca_denoise = PCA(n_components=20)
pca_denoise.fit(X_scaled)

noisy_proj = pca_denoise.transform([noisy_scaled])
denoised_scaled = pca_denoise.inverse_transform(noisy_proj)[0]

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(clean_digit.reshape(8, 8), cmap='gray')
axes[0].set_title('Original Clean Digit', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(noisy_digit.reshape(8, 8), cmap='gray')
axes[1].set_title('Noisy Digit', fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(denoised_scaled.reshape(8, 8), cmap='gray')
axes[2].set_title('Denoised (PCA, 20 components)', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"Noise level (std): {np.std(noise):.2f}")
print(f"PCA removed high-frequency noise by projecting onto 20 principal components.")

## Part 9: PCA in Machine Learning Pipeline

### Use Case: Speedup Classification with Dimensionality Reduction

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import time

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# Test different numbers of components
n_components_range = [5, 10, 20, 30, 40, 50, 64]
accuracies = []
times = []

for n_comp in n_components_range:
    # Apply PCA
    if n_comp < 64:
        pca = PCA(n_components=n_comp)
        X_train_pca = pca.fit_transform(X_train)
        X_test_pca = pca.transform(X_test)
    else:
        X_train_pca = X_train
        X_test_pca = X_test
    
    # Train classifier
    start = time.time()
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train_pca, y_train)
    y_pred = clf.predict(X_test_pca)
    elapsed = time.time() - start
    
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    times.append(elapsed)
    
    print(f"Components: {n_comp:2d} | Accuracy: {acc:.4f} | Time: {elapsed:.4f}s")

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(n_components_range, accuracies, 'o-', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Components', fontsize=12)
ax1.set_ylabel('Test Accuracy', fontsize=12)
ax1.set_title('Classification Accuracy vs Dimensionality', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)

ax2.plot(n_components_range, times, 'o-', linewidth=2, markersize=8, color='orange')
ax2.set_xlabel('Number of Components', fontsize=12)
ax2.set_ylabel('Training Time (s)', fontsize=12)
ax2.set_title('Training Time vs Dimensionality', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary and Key Takeaways

### What We Learned

1. **PCA is SVD on centered data**: $\tilde{X} = U\Sigma V^T$, principal components are $V$

2. **Dimensionality reduction**: Project high-dim data to low-dim while preserving variance

3. **Visualization**: PCA enables plotting high-dimensional data in 2D/3D

4. **Choosing k**: Use cumulative explained variance (e.g., 90% threshold)

5. **Applications in ML**:
   - Preprocessing for faster training
   - Noise reduction
   - Feature extraction
   - Visualization and EDA

### When to Use PCA

✅ **Use PCA when:**
- You have many correlated features
- You need faster training times
- You want to visualize high-dim data
- You want to remove noise

❌ **Don't use PCA when:**
- Features are already uncorrelated
- You need interpretable features (PCs are linear combinations)
- Your data is sparse (PCA densifies it)
- Non-linear relationships are important (use kernel PCA or autoencoders)

### Connection to Other ML Concepts

- **Autoencoders**: Non-linear PCA using neural networks
- **t-SNE/UMAP**: Non-linear alternatives for visualization
- **Factor Analysis**: Probabilistic version of PCA
- **SVD**: PCA is a special case of SVD

### Next Steps

- Move to **Project 3: Recommender Systems** to see matrix factorization in action
- Explore **Project 4: LASSO** for sparse feature selection (alternative to PCA)
- Try kernel PCA for non-linear dimensionality reduction

---

**Excellent work!** You now understand PCA and can apply it to real ML problems. 🎉